In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile # To handle zip files
import shutil # To remove directories if re-running

# --- 1. 🛠️ USER CONFIGURATION ---
IMG_WIDTH = 128     # Target width for resizing images (e.g., 64, 128, 256)
IMG_HEIGHT = 128    # Target height for resizing images
IMAGE_SIZE = (IMG_WIDTH, IMG_HEIGHT)
# COLOR_MODE: 'rgb' for color images, 'grayscale' for black and white.
# Using 'rgb' is generally better for CNNs unless you have a specific reason for grayscale.
COLOR_MODE = 'rgb'
CHANNELS = 3 if COLOR_MODE == 'rgb' else 1

# General training parameters
BATCH_SIZE = 32
EPOCHS = 25         # Adjust based on dataset size and complexity (15-50 is common)
LEARNING_RATE = 0.001

# Dataset parameters
DATASET_ZIP_NAME = 'my_cnn_dataset.zip' # The name of the zip file you will upload
DATASET_EXTRACT_PATH = 'extracted_cnn_dataset' # Folder where dataset will be extracted

# --- 2. 📂 Dataset Upload and Preparation ---
print(f"🚀 Starting CNN Image Classification")
print(f"Expecting images of size: {IMAGE_SIZE}, Color: {COLOR_MODE}")

from google.colab import files
print(f"\nPlease upload your dataset ZIP file named '{DATASET_ZIP_NAME}'")
print(f"This ZIP file should contain folders for each class (e.g., a 'dogs' folder and a 'cats' folder).")
uploaded = files.upload()

if DATASET_ZIP_NAME in uploaded:
    print(f"\n✅ '{DATASET_ZIP_NAME}' uploaded successfully!")
    # Clean up old extracted content if it exists
    if os.path.exists(DATASET_EXTRACT_PATH):
        print(f"🧹 Cleaning up existing directory: {DATASET_EXTRACT_PATH}")
        shutil.rmtree(DATASET_EXTRACT_PATH)
    os.makedirs(DATASET_EXTRACT_PATH, exist_ok=True)

    with zipfile.ZipFile(DATASET_ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(DATASET_EXTRACT_PATH)
    print(f"🗂️ Dataset extracted to '{DATASET_EXTRACT_PATH}'")

    # Attempt to find the correct dataset root (where class folders are)
    extracted_items = os.listdir(DATASET_EXTRACT_PATH)
    if not extracted_items:
        print(f"❌ Error: The extracted directory '{DATASET_EXTRACT_PATH}' is empty.")
        exit()

    dataset_dir = DATASET_EXTRACT_PATH
    # If the zip file contains a single root folder, navigate into it
    if len(extracted_items) == 1 and os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])):
        dataset_dir = os.path.join(DATASET_EXTRACT_PATH, extracted_items[0])

    print(f"🔍 Using dataset directory: {dataset_dir}")

    class_folders = [d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))]
    if not class_folders or len(class_folders) < 2: # Need at least 2 classes
        print(f"❌ ERROR: Dataset directory '{dataset_dir}' must contain at least two subfolders (classes).")
        exit()
else:
    print(f"❌ ERROR: '{DATASET_ZIP_NAME}' not found. Please upload the correct file.")
    exit()

# --- 3. 🖼️ Load Data with Preprocessing ---
print("\n⏳ Loading and preprocessing data...")

try:
    # Load images, resize, and set color mode
    # image_dataset_from_directory generates labels as one-hot encoded vectors
    # when label_mode='categorical'
    train_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2, # Reserve 20% of data for validation
        subset="training",
        seed=123,             # Seed for shuffling and splitting consistency
        image_size=IMAGE_SIZE,
        color_mode=COLOR_MODE,
        batch_size=BATCH_SIZE,
        label_mode='categorical' # Suitable for 'categorical_crossentropy' loss
    )

    validation_dataset = image_dataset_from_directory(
        dataset_dir,
        validation_split=0.2,
        subset="validation",
        seed=123,
        image_size=IMAGE_SIZE,
        color_mode=COLOR_MODE,
        batch_size=BATCH_SIZE,
        label_mode='categorical'
    )
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Please ensure your dataset_dir contains subdirectories for each class and images are valid.")
    exit()

class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found classes: {class_names} (Number of classes: {num_classes})")

if num_classes < 2:
    print(f"❌ Error: Classification requires at least 2 classes. Found only {num_classes}.")
    exit()

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# --- 4. 🧠 Build the CNN Model ---
print(f"\n⏳ Building CNN model...")

model = models.Sequential([
    # Input layer: Rescale pixel values from [0, 255] to [0, 1]
    layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),

    # Convolutional Block 1
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Convolutional Block 2
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Convolutional Block 3
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # (Optional) Convolutional Block 4 - can add more for deeper networks
    # layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
    # layers.MaxPooling2D((2, 2)),

    # Flatten the results to feed into a DNN
    layers.Flatten(),

    # Dense (Fully Connected) Layers
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5), # Dropout for regularization to prevent overfitting

    # Output layer
    # num_classes will be 2 for binary (e.g., dog/cat), or more for multi-class
    layers.Dense(num_classes, activation='softmax')
])

# --- 5. ⚙️ Compile the Model ---
print("\n⚙️ Compiling the model...")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy', # Use for one-hot encoded labels
    metrics=['accuracy']
)
model.summary() # Print the model structure

# --- 6. 🚀 Train the Model ---
print("\n🚀 Starting model training...")
# Optional: Add callbacks like EarlyStopping
# early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1 # Show progress bar during training
    # callbacks=[early_stopping] # Uncomment to use early stopping
)

# --- 7. 📊 Evaluate and Plot ---
print("\n⚖️ Evaluating model and plotting history...")
# If EarlyStopping with restore_best_weights was used, this evaluates the best model.
val_loss, val_accuracy = model.evaluate(validation_dataset, verbose=0)
print(f"\n✅ Final Validation Loss: {val_loss:.4f}")
print(f"🎯 Final Validation Accuracy: {val_accuracy*100:.2f}%")

# Plot training & validation accuracy and loss values
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss_hist = history.history['loss'] # Renamed to avoid conflict with 'loss' from evaluate
val_loss_hist = history.history['val_loss'] # Renamed
epochs_range = range(len(acc)) # Use actual number of epochs run

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('CNN - Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss_hist, label='Training Loss')
plt.plot(epochs_range, val_loss_hist, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('CNN - Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.tight_layout()
plt.show()

# --- 8. 🔮 Predict on a New Image ---
def predict_single_new_image_cnn(trained_model, class_names_list, img_size_tuple, color_mode_str):
    print("\n🖼️ Upload an image for prediction:")
    uploaded_img_dict = files.upload()

    if not uploaded_img_dict:
        print("No file uploaded.")
        return

    file_path = list(uploaded_img_dict.keys())[0]

    try:
        # Load the image, ensuring it matches the model's expected input size and color mode
        img = tf.keras.preprocessing.image.load_img(
            file_path,
            target_size=img_size_tuple,
            color_mode=color_mode_str
        )
        # Convert PIL image to NumPy array for display
        img_for_display = tf.keras.preprocessing.image.img_to_array(img)

        # Preprocess for model input (no separate normalization needed if Rescaling layer is in model)
        img_for_model_input = tf.keras.preprocessing.image.img_to_array(img)
        img_batch = tf.expand_dims(img_for_model_input, 0) # Create a batch (shape: [1, height, width, channels])

        predictions_output = trained_model.predict(img_batch) # Output shape: (1, num_classes)

        # For softmax output
        predicted_index = np.argmax(predictions_output[0]) # Index of the highest probability
        confidence = np.max(predictions_output[0]) * 100    # Highest probability
        predicted_class_name = class_names_list[predicted_index]

        # Displaying the image
        plt.figure()
        if color_mode_str == 'grayscale':
            plt.imshow(img_for_display.squeeze(), cmap='gray') # Squeeze for grayscale display
        else:
            plt.imshow(img_for_display.astype(np.uint8)) # Convert to uint8 for RGB display
        plt.title(f"Predicted: {predicted_class_name} ({confidence:.2f}%)")
        plt.axis("off")
        plt.show()

        print(f"The image is predicted as: '{predicted_class_name}' with {confidence:.2f}% confidence.")
        print(f"Raw prediction scores (probabilities for each class {class_names}): {predictions_output[0]}")

    except Exception as e:
        print(f"❌ Error processing or predicting image: {e}")

# Make a prediction on a new image using the trained CNN model
predict_single_new_image_cnn(model, class_names, IMAGE_SIZE, COLOR_MODE)

print("\n🎉 --- End of CNN Classification Script --- 🎉")